# Les files — Exercice 6 : une pile avec deux files (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** implémenter une pile (LIFO) **à l'aide** de files (FIFO), invariant d'une classe, coût des opérations.

Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

---
# MISE EN PLACE

> Dans la suite, on utilisera exclusivement uniquement des objets de la classe File définie dans l'exercice 1.
> (Interdit d'utiliser le type list de python.)

On importe la classe `File` de l'exercice 1 : le fichier `file.py` doit être **dans le même dossier** que ce notebook.

**Aucune liste python, aucune boucle `for`** : on ne manipule les files qu'avec `enfile`, `defile`, `est_vide` (et `taille`). Pour les vérifications, on fabrique les files par des `enfile` successifs, et on compare leur **affichage** au texte attendu.

Une file s'affiche sous la forme `File(['a', 'b', 'c'])` : la **tête** est le premier élément (à gauche, c'est lui qu'on défile), la **queue** est le dernier (à droite, c'est là qu'on enfile).

In [1]:
from file import File

f = File()
f.enfile("a")
f.enfile("b")
f.enfile("c")
print(f)

File(['a', 'b', 'c'])


---
# QUESTION

#### 📝 ÉNONCÉ

> Écrire l'implémentation d'une classe `Pile` qui utilise deux files.
> On rappelle que les méthodes à définir sont : `empile`, `depile`, `est_vide` et `taille`.
> Indications :
> - une des deux files est toujours vide ;
> - la pile est vide lorsque les deux files sont vides ;
> - pour empiler un élément, on l'enfile dans celle des deux files qui est non vide ;
> - pour dépiler un élément, on transfère les éléments de la file non vide vers l'autre file, à l'exception du dernier élément, qui est renvoyé.

#### ✅ RÉSOLUTION

In [2]:
class Pile:

    def __init__(self):
        self.f1 = File()
        self.f2 = File()

    def est_vide(self):
        return self.f1.est_vide() and self.f2.est_vide()

    def taille(self):
        return self.f1.taille() + self.f2.taille()

    def empile(self, el):
        if self.f2.est_vide():
            self.f1.enfile(el)
        else:
            self.f2.enfile(el)

    def depile(self):
        if self.est_vide():
            raise IndexError("depile() sur une pile vide")
        if self.f2.est_vide():
            pleine, vide = self.f1, self.f2
        else:
            pleine, vide = self.f2, self.f1
        while pleine.taille() > 1:
            vide.enfile(pleine.defile())
        return pleine.defile()

*Vérification (non demandée) : le LIFO, l'alternance empile / dépile, l'invariant et le cas d'erreur.*

In [3]:
p = Pile()
assert p.est_vide() and p.taille() == 0
p.empile(1)
p.empile(2)
p.empile(3)
assert p.taille() == 3
assert p.depile() == 3           # le dernier empilé sort en premier
p.empile(4)
assert p.depile() == 4
assert p.depile() == 2
assert p.depile() == 1
assert p.est_vide()

# invariant : après chaque opération, au moins une des deux files est vide
x = 0
while x < 10:
    p.empile(x)
    assert p.f1.est_vide() or p.f2.est_vide()
    x = x + 1
x = 9
while not p.est_vide():
    assert p.depile() == x
    assert p.f1.est_vide() or p.f2.est_vide()
    x = x - 1

try:
    p.depile()
except IndexError as e:
    print("Erreur bien déclenchée :", e)
print("Tous les tests passent")

Erreur bien déclenchée : depile() sur une pile vide
Tous les tests passent


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Rappel :** une pile fonctionne en **LIFO** (*Last In, First Out*) : on dépile le **dernier** empilé. Or une file ne donne accès qu'à son **premier** élément : il faut faire défiler tout le monde pour atteindre le dernier.
- **Les indications, une par une :**
  - « une des deux files est toujours vide » : c'est un **invariant**, vrai après chaque appel. Tous les éléments sont dans **une seule** file, dans l'ordre d'empilement ;
  - `est_vide` : les deux files sont vides ; `taille` : la somme des deux tailles (l'une vaut 0) ;
  - `empile` : on enfile dans la file non vide (si les deux sont vides, on prend `f1`) ;
  - `depile` : on transfère tout **sauf le dernier** dans l'autre file, puis on défile et renvoie ce dernier. Les éléments sont alors dans l'autre file : l'invariant reste vrai.
- **Exemple :** on empile 1, 2, 3, puis on dépile.

| Étape | `f1` | `f2` |
|:---|:---|:---|
| après les empilements | `1 2 3` | |
| transfert de 1 et 2 | `3` | `1 2` |
| on défile et renvoie 3 | | `1 2` |

- **Coût :** `empile` fait un seul `enfile`, mais `depile` déplace **tous** les éléments sauf un : son coût est proportionnel à la taille de la pile.
- ⚠️ **Erreurs fréquentes :** transférer **tous** les éléments, y compris le dernier (on ne sait plus lequel renvoyer) ; toujours enfiler dans `f1` (après un `depile`, les éléments sont dans `f2` et l'invariant est cassé).